In [1]:
from mip import *
import pandas as pd
from icecream import ic
import numpy as np

import io
from functools import reduce
import re
import math
from dataclasses import dataclass
from typing import Literal
from collections import defaultdict

In [2]:
pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

In [3]:
recipes_df = pd.read_excel('datasheet.ods', sheet_name='配方', index_col=1)

recipes_df = recipes_df[~recipes_df['禁用']].drop(columns=['禁用'])

is_powergen = recipes_df['工艺'].str.startswith('热能池')
powergen_df = recipes_df[is_powergen].reset_index()
is_metastorage = recipes_df['工艺'].str.startswith('超库存传输')
metastorage_df = recipes_df[is_metastorage].reset_index()
recipes_df = recipes_df[~(is_powergen | is_metastorage)].reset_index()

display(recipes_df, powergen_df, metastorage_df)

,功率,工艺,原料1,原料1每分钟消耗,原料2,原料2每分钟消耗,产物1,产物1每分钟产率,产物2,产物2每分钟产率
0,-10,水泵,NaN,NaN,NaN,NaN,水,60.0,NaN,NaN
1,-5,精炼炉,蓝铁矿,-30.0,NaN,NaN,蓝铁块,30.0,NaN,NaN
2,-5,精炼炉,紫晶矿,-30.0,NaN,NaN,紫晶纤维,30.0,NaN,NaN
3,-5,精炼炉,源矿,-30.0,NaN,NaN,晶体外壳,30.0,NaN,NaN
4,-5,精炼炉,致密晶体粉末,-30.0,NaN,NaN,密制晶体,30.0,NaN,NaN
5,-5,精炼炉,致密蓝铁粉末,-30.0,NaN,NaN,钢块,30.0,NaN,NaN
6,-5,精炼炉,高晶粉末,-30.0,NaN,NaN,高晶纤维,30.0,NaN,NaN
7,-5,精炼炉,致密碳粉末,-30.0,NaN,NaN,稳定碳块,30.0,NaN,NaN
8,-5,精炼炉,芽针,-30.0,NaN,NaN,碳块,60.0,NaN,NaN
9,-5,精炼炉,赤铜矿,-30.0,水,-30.0,赤铜块,30.0,污水,30.0


,功率,工艺,原料1,原料1每分钟消耗,原料2,原料2每分钟消耗,产物1,产物1每分钟产率,产物2,产物2每分钟产率
0,50,热能池-源矿,源矿,-7.5,NaN,NaN,NaN,NaN,NaN,NaN
1,220,热能池-低容谷地电池,低容谷地电池,-1.5,NaN,NaN,NaN,NaN,NaN,NaN
2,420,热能池-中容谷地电池,中容谷地电池,-1.5,NaN,NaN,NaN,NaN,NaN,NaN
3,1100,热能池-高容谷地电池,高容谷地电池,-1.5,NaN,NaN,NaN,NaN,NaN,NaN
4,1600,热能池-低容武陵电池,低容武陵电池,-1.5,NaN,NaN,NaN,NaN,NaN,NaN
5,3200,热能池-中容武陵电池,中容武陵电池,-1.5,NaN,NaN,NaN,NaN,NaN,NaN


,功率,工艺,原料1,原料1每分钟消耗,原料2,原料2每分钟消耗,产物1,产物1每分钟产率,产物2,产物2每分钟产率
0,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.016667,NaN,NaN,蓝铁矿,0.016667,NaN,NaN
1,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.016667,NaN,NaN,紫晶矿,0.016667,NaN,NaN
2,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.016667,NaN,NaN,源矿,0.016667,NaN,NaN
3,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.333333,NaN,NaN,低容谷地电池,0.016667,NaN,NaN
4,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.333333,NaN,NaN,中容谷地电池,0.016667,NaN,NaN
5,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.833333,NaN,NaN,高容谷地电池,0.016667,NaN,NaN


In [4]:
materials = reduce(
    set.union,
    (
        set(recipes_df[col])
        for col in recipes_df.columns
        if re.match(r'^(原料|产物)\d?$', col)
    )
) - {np.nan, math.nan}
materials = list(materials)

_=ic(materials)

ic| materials: ['壤晶废液',
                '高晶零件',
                '晶体粉末',
                '芽针溶液',
                '致密蓝铁粉末',
                '晶体外壳',
                '赤铜瓶',
                '低容谷地电池',
                '惰性壤晶废液',
                '砂叶',
                '中容武陵电池',
                '钢制零件',
                '壤晶',
                '紫晶质瓶',
                '碳粉末',
                '中容谷地电池',
                '蓝铁矿',
                '致密源石粉末',
                '息壤装备原件',
                '污水',
                '致密碳粉末',
                '高晶纤维',
                '赤铜矿',
                '蓝铁瓶-芽针溶液',
                '蓝铁瓶',
                '息壤',
                '芽针',
                '紫晶粉末',
                '碳块',
                '芽针针剂',
                '稳定碳块',
                '芽针粉末',
                '钢块',
                '蓝铁块',
                '液化息壤',
                '源矿粉末',
                '铁制零件',
                '致密晶体粉末',
                '低容武陵电池',
                '蓝铁粉末',
                '紫晶纤维',
             

In [5]:
INIT_POWER = 200.0
EXTERN_POWER = -700.0  # 基地外消耗
# INIT_POWER = np.inf # 手动搬电池
INIT_MATERIAL = {
    '源矿': 480.0,
    '紫晶矿': 0.0,
    '蓝铁矿': 90.0,
    '赤铜矿': 120.0,
}
METASTORAGE_QUOTA_MIN = 1500.0 / 60.0
XIRANG_OVEN_LIMIT = 4
VALUES = {
    # '低容武陵电池': 25.0,
    '中容武陵电池': 54.0,
    '芽针针剂': 16.0,
    '优质芽针针剂': 22.0,
    '息壤': 1.0,
    '赤铜零件': 1.0,
}
TARGET_VALUE = 605  # 604.8
DISPOSAL_POWER = -50.0  # 废水处理功率
RESOLUTION = 8.0  # 粒度, 最小速率变化量为 1/x
MERGABLE_MACHINE = [
    # '精炼炉',
    # '粉碎机',
    # '配件机',
    # '塑形机',
    # '双种植采种循环',
    # '双种植采种循环-液体',
    # '单种植采种循环',
    # '单种植采种循环-液体',
    # '装备原件机',
    # '灌装机',
    # '封装机',
    # '研磨机',
    # '反应池',
    # '天有洪炉',
    # '拆解机',
]

n_recipes = recipes_df.shape[0]
n_powergen = powergen_df.shape[0]
n_metastorage = metastorage_df.shape[0]

m = Model(sense=MAXIMIZE)
x_recipes = m.add_var_tensor((n_recipes, ), 'x_recipes', lb=0.0, ub=INF, var_type=INTEGER)
x_recipes_ceil = m.add_var_tensor((n_recipes, ), 'x_recipes_ceil', lb=0.0, ub=INF, var_type=INTEGER)
m += (x_recipes_ceil >= x_recipes / RESOLUTION - 1e-8)
x_powergen = m.add_var_tensor((n_powergen, ), 'x_powergen', lb=0.0, ub=INF, var_type=INTEGER)
x_metastorage = m.add_var_tensor((n_metastorage, ), 'x_metastorage', lb=0.0, ub=1.0, var_type=INTEGER)


tot_material = defaultdict(lambda: 0)
for mat, cnt in INIT_MATERIAL.items():
    tot_material[mat] += cnt

# ===== 统计生产配方的材料增减 =====
for i, row in enumerate(recipes_df.itertuples()):
    mat1, mat2, prd1, prd2 = getattr(row, '原料1'), getattr(row, '原料2'), getattr(row, '产物1'), getattr(row, '产物2')
    x_i = x_recipes[i] / RESOLUTION
    if mat1 not in (np.nan, math.nan):
        tot_material[mat1] += x_i * getattr(row, '原料1每分钟消耗')
    if mat2 not in (np.nan, math.nan):
        tot_material[mat2] += x_i * getattr(row, '原料2每分钟消耗')
    if prd1 not in (np.nan, math.nan):
        tot_material[prd1] += x_i * getattr(row, '产物1每分钟产率')
    if prd2 not in (np.nan, math.nan):
        tot_material[prd2] += x_i * getattr(row, '产物2每分钟产率')

# ===== 统计发电配方的材料增减 =====
for i, row in enumerate(powergen_df.itertuples()):
    mat1, mat2, prd1, prd2 = getattr(row, '原料1'), getattr(row, '原料2'), getattr(row, '产物1'), getattr(row, '产物2')
    x_i = x_powergen[i] / RESOLUTION
    if mat1 not in (np.nan, math.nan):
        tot_material[mat1] += x_i * getattr(row, '原料1每分钟消耗')
    if mat2 not in (np.nan, math.nan):
        tot_material[mat2] += x_i * getattr(row, '原料2每分钟消耗')
    if prd1 not in (np.nan, math.nan):
        tot_material[prd1] += x_i * getattr(row, '产物1每分钟产率')
    if prd2 not in (np.nan, math.nan):
        tot_material[prd2] += x_i * getattr(row, '产物2每分钟产率')

# ===== 统计超库存传输的材料增减 =====
for i, row in enumerate(metastorage_df.itertuples()):
    prd1, prd2 = getattr(row, '产物1'), getattr(row, '产物2')
    x_i = x_metastorage[i]
    if prd1 not in (np.nan, math.nan):
        tot_material[prd1] += x_i * (METASTORAGE_QUOTA_MIN / -getattr(row, '原料1每分钟消耗') * getattr(row, '产物1每分钟产率'))
    if prd2 not in (np.nan, math.nan):
        tot_material[prd2] += x_i * (METASTORAGE_QUOTA_MIN / -getattr(row, '原料1每分钟消耗') * getattr(row, '产物2每分钟产率'))

# ===== 材料总量不小于零 =====
for mat, cnt in tot_material.items():
    if mat in ['污水', '壤晶废液', '惰性壤晶废液'] \
        or mat in ['赤铜零件', '赤铜块'] or '赤铜瓶' in mat:
        # 废水清零，赤铜不堵塞
        m += (tot_material[mat] == 0, mat)
    else:
        m += (cnt >= 0, mat)

# ===== 废水处理无小数，可能可以细分但过于麻烦了 =====
for i in recipes_df[recipes_df['工艺'] == '废水处理机'].index:
    m += (x_recipes[i] / RESOLUTION == x_recipes_ceil[i])

# ===== 天有洪炉上限 =====
tot_xirang_oven = xsum(
    x_recipes_ceil[
        list(recipes_df[recipes_df['工艺'] == '天有洪炉'].index)
    ]
)
m += (tot_xirang_oven <= XIRANG_OVEN_LIMIT, '天有洪炉')

# ===== 超库存传输单选 =====
m += (xsum(x_metastorage) <= 1, '超库存传输')

# ===== 调度券量不小于阈值 =====
tot_value = xsum(tot_material[mat] * val for mat, val in VALUES.items())
m += tot_value >= TARGET_VALUE

# ===== 总功率不小于零 =====
tot_power = INIT_POWER + EXTERN_POWER + xsum(
    x_recipes_ceil * recipes_df['功率']
) + xsum(
    x_powergen / RESOLUTION * powergen_df['功率']
)
m += (tot_power >= 0, 'tot_power')

m += tot_material['赤铜装备原件'] >= 1.0
m += tot_material['息壤装备原件'] >= 0.5
m.objective = tot_material['赤铜装备原件'] + tot_material['息壤装备原件']
# m.objective = tot_material['赤铜装备原件']
# m.objective = tot_material['息壤装备原件']
m.objective += tot_power / 1e4  # 防止大分辨率下的额外机器
# m.objective -= xsum(x_recipes_ceil - x_recipes / RESOLUTION) / 1e2  # 略微减少非整数速率，容易导致中间产物过多

In [6]:
m.optimize(max_seconds=30)

Welcome to the CBC MILP Solver 
Version: Trunk
Build Date: Oct 24 2021 

Starting solution of the Linear programming relaxation problem using Primal Simplex

Coin0506I Presolve 71 (-50) rows, 81 (-41) columns and 236 (-101) elements
Clp0029I End of values pass after 56 iterations
Clp0014I Perturbing problem by 0.001% of 4.8720447 - largest nonzero change 3.0473258e-05 ( 0.001199713%) - largest zero change 2.8981471e-05
Clp0000I Optimal - objective value 2.9465968
Coin0511I After Postsolve, objective 2.9465968, infeasibilities - dual 0 (0), primal 0 (0)
Clp0032I Optimal objective 2.946596785 - 77 iterations time 0.012, Presolve 0.00, Idiot 0.01

Starting MIP optimization


<OptimizationStatus.OPTIMAL: 0>

In [7]:
print('调度券分钟产量', tot_value.x, '日产量', tot_value.x * 60 * 24, '周产量', tot_value.x * 60 * 24 * 7)
print('息壤装备原件分钟产量', tot_material['息壤装备原件'].x, '日产量', tot_material['息壤装备原件'].x * 60 * 24, '周产量', tot_material['息壤装备原件'].x * 60 * 24 * 7)
print('赤铜装备原件分钟产量', tot_material['赤铜装备原件'].x, '日产量', tot_material['赤铜装备原件'].x * 60 * 24, '周产量', tot_material['赤铜装备原件'].x * 60 * 24 * 7)
result_tot_power_machine = np.sum(np.array([u.x for u in x_recipes_ceil]) * recipes_df['功率'])
print('产线机器耗电', result_tot_power_machine)
print('最大允许耗电', result_tot_power_machine + EXTERN_POWER)

print('超库存传输', metastorage_df.loc[(u.x == 1.0 for u in x_metastorage)]['产物1'].item())

def format_float(raw: float):
    if raw == 0:
        return "0"
    x, y = raw.as_integer_ratio()
    if x % y == 0:
        return str(x // y)
    if x < y:
        return f"{raw} ({x}/{y})"
    a, x = x // y, x % y
    return f"{raw} ({a} {x}/{y})"

result_df =pd.concat([
    pd.concat([
        recipes_df,
        pd.DataFrame(
            {'速率': format_float(u.x / RESOLUTION), '建造数量': v.x}
            for u, v in zip(x_recipes, x_recipes_ceil)
        ),
    ], axis=1),
    pd.concat([
        powergen_df,
        pd.DataFrame(
            {'速率': format_float(u.x / RESOLUTION), '建造数量': math.ceil(u.x / RESOLUTION)}
            for u in x_powergen
        ),
    ], axis=1)
])
result_df = result_df[result_df['建造数量'] != 0]
display(result_df)


None

调度券分钟产量 609.0 日产量 876960.0 周产量 6138720.0
息壤装备原件分钟产量 0.75 日产量 1080.0 周产量 7560.0
赤铜装备原件分钟产量 1.5 日产量 2160.0 周产量 15120.0
产线机器耗电 -2815.0
最大允许耗电 -3515.0
超库存传输 蓝铁矿


,功率,工艺,原料1,原料1每分钟消耗,原料2,原料2每分钟消耗,产物1,产物1每分钟产率,产物2,产物2每分钟产率,速率,建造数量
0,-10,水泵,NaN,NaN,NaN,NaN,水,60.0,NaN,NaN,8.625 (8 5/8),9.0
1,-5,精炼炉,蓝铁矿,-30.0,NaN,NaN,蓝铁块,30.0,NaN,NaN,3.75 (3 3/4),4.0
3,-5,精炼炉,源矿,-30.0,NaN,NaN,晶体外壳,30.0,NaN,NaN,0.5 (1/2),1.0
4,-5,精炼炉,致密晶体粉末,-30.0,NaN,NaN,密制晶体,30.0,NaN,NaN,0.25 (1/4),1.0
7,-5,精炼炉,致密碳粉末,-30.0,NaN,NaN,稳定碳块,30.0,NaN,NaN,8,8.0
8,-5,精炼炉,芽针,-30.0,NaN,NaN,碳块,60.0,NaN,NaN,4,4.0
9,-5,精炼炉,赤铜矿,-30.0,水,-30.0,赤铜块,30.0,污水,30.0,4,4.0
11,-5,粉碎机,蓝铁块,-30.0,NaN,NaN,蓝铁粉末,30.0,NaN,NaN,1.5 (1 1/2),2.0
13,-5,粉碎机,源矿,-30.0,NaN,NaN,源石粉末,30.0,NaN,NaN,13.5 (13 1/2),14.0
14,-5,粉碎机,碳块,-30.0,NaN,NaN,碳粉末,60.0,NaN,NaN,8,8.0
